<a href="https://colab.research.google.com/github/ajsarsva/video-captioning-thesis/blob/main/notebooks/colab_setup_5.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
for folder in folders:
    path = os.path.join(base, folder)
    print(f"{path} — exists: {os.path.exists(path)}")

/content/drive/MyDrive/thesis-data/videos — exists: True
/content/drive/MyDrive/thesis-data/frames — exists: True
/content/drive/MyDrive/thesis-data/captions — exists: True
/content/drive/MyDrive/thesis-data/results — exists: True


In [1]:
!git clone https://github.com/ajsarsva/video-captioning-thesis.git

Cloning into 'video-captioning-thesis'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 12 (delta 1), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), done.
Resolving deltas: 100% (1/1), done.


In [2]:
# Install kaggle
!pip install kaggle -q

In [4]:
import json
import os

kaggle_creds = {
    "username": "ajsarsva",  # your Kaggle username
    "key": "KGAT_6364ab60874f270326ac670e5e1f25ff"  # your actual token
}

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump(kaggle_creds, f)

os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

print("kaggle.json created successfully!")

kaggle.json created successfully!


In [8]:
import shutil

# Save to Drive once
shutil.copy('/root/.kaggle/kaggle.json', '/content/drive/MyDrive/kaggle.json')
print("Saved to Drive!")

Saved to Drive!


In [9]:
from google.colab import files
files.upload()  # upload your kaggle.json here

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username": "ajsarsva", "key": "KGAT_6364ab60874f270326ac670e5e1f25ff"}'}

In [10]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [11]:
# Download MSR-VTT dataset
!kaggle datasets download -d vishnutheepb/msrvtt -p /content/msrvtt_raw

Dataset URL: https://www.kaggle.com/datasets/vishnutheepb/msrvtt
License(s): unknown
100% 4.26G/4.26G [00:44<00:00, 104MB/s]



In [12]:
!unzip -q /content/msrvtt_raw/msrvtt.zip -d /content/msrvtt_extracted
!ls /content/msrvtt_extracted

TrainValVideo


In [14]:
for root, dirs, files in os.walk('/content/msrvtt_extracted'):
    level = root.replace('/content/msrvtt_extracted', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:  # only show top 2 levels
        for f in files[:5]:  # show first 5 files per folder
            print(f'{indent}  {f}')

msrvtt_extracted/
  TrainValVideo/
    video6920.mp4
    video4799.mp4
    video448.mp4
    video3352.mp4
    video1230.mp4


In [23]:
!kaggle datasets download -d mrandri19/msr-vtt -p /content/msrvtt_full_raw

Dataset URL: https://www.kaggle.com/datasets/mrandri19/msr-vtt
License(s): unknown
100% 6.11G/6.11G [01:14<00:00, 88.4MB/s]



In [24]:
!unzip -l /content/msrvtt_full_raw/msr-vtt.zip | head -50

Archive:  /content/msrvtt_full_raw/msr-vtt.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
 20851292  2025-08-17 18:34   data/MSRVTT/MSRVTT/annotation/MSR_VTT.json
    20245  2025-08-17 18:34   data/MSRVTT/MSRVTT/high-quality/structured-symlinks/jsfusion_val_caption_idx.pkl
 26152963  2025-08-17 18:34   data/MSRVTT/MSRVTT/high-quality/structured-symlinks/raw-captions.pkl
       50  2025-08-17 18:34   data/MSRVTT/MSRVTT/high-quality/structured-symlinks/test_list_1.txt
    29900  2025-08-17 18:34   data/MSRVTT/MSRVTT/high-quality/structured-symlinks/test_list_full.txt
     9886  2025-08-17 18:34   data/MSRVTT/MSRVTT/high-quality/structured-symlinks/test_list_miech.txt
       50  2025-08-17 18:34   data/MSRVTT/MSRVTT/high-quality/structured-symlinks/test_list_mini.txt
    29860  2025-08-17 18:34   data/MSRVTT/MSRVTT/high-quality/structured-symlinks/test_list_only20capts.txt
    11836  2025-08-17 18:34   data/MSRVTT/MSRVTT/high-quality/structured-symlinks/train_li

In [25]:
!unzip -j /content/msrvtt_full_raw/msr-vtt.zip \
    "data/MSRVTT/MSRVTT/annotation/MSR_VTT.json" \
    -d /content/msrvtt_annotation

Archive:  /content/msrvtt_full_raw/msr-vtt.zip
  inflating: /content/msrvtt_annotation/MSR_VTT.json  


In [26]:
import json

with open('/content/msrvtt_annotation/MSR_VTT.json', 'r') as f:
    data = json.load(f)

print("Keys:", data.keys())
print("\nFirst video info:", data['videos'][0])
print("\nFirst caption:", data['sentences'][0])

Keys: dict_keys(['info', 'images', 'licenses', 'type', 'annotations'])


KeyError: 'videos'

In [27]:
print("Keys:", data.keys())
print("\nFirst image info:", data['images'][0])
print("\nFirst annotation:", data['annotations'][0])

Keys: dict_keys(['info', 'images', 'licenses', 'type', 'annotations'])

First image info: {'id': 'video0'}

First annotation: {'caption': 'a cartoon animals runs through an ice cave in a video game', 'id': 0, 'image_id': 'video2960'}


In [28]:
# How many videos and captions total?
print("Total videos:", len(data['images']))
print("Total captions:", len(data['annotations']))

# See a few videos
print("\nFirst 3 videos:")
for v in data['images'][:3]:
    print(v)

# See all captions for one specific video
print("\nAll captions for video0:")
for ann in data['annotations']:
    if ann['image_id'] == 'video0':
        print(ann['caption'])


Total videos: 10000
Total captions: 199994

First 3 videos:
{'id': 'video0'}
{'id': 'video1'}
{'id': 'video2'}

All captions for video0:
a car is shown
a group is dancing
a man drives a vehicle through the countryside
a man drives down the road in an audi
a man driving a car
a man is driving a car
a man is driving down a road
a man is driving in a car as part of a commercial
a man is driving
a man riding the car speedly in a narrow road
a man showing the various features of a car
a man silently narrates his experience driving an audi
a person is driving his car around curves in the road
a person telling about a car
guy driving a car down the road
man talking about a car while driving
the man drives the car
the man driving the audi as smooth as possible
a man is driving
guy driving a car down the road


In [29]:
import json

# Get first 50 video IDs
dev_video_ids = set(f'video{i}' for i in range(50))

# Filter annotations for only these 50 videos
dev_annotations = [ann for ann in data['annotations']
                   if ann['image_id'] in dev_video_ids]

dev_videos = [v for v in data['images']
              if v['id'] in dev_video_ids]

print(f"Dev videos: {len(dev_videos)}")
print(f"Dev captions: {len(dev_annotations)}")  # should be ~1000

# Save it
dev_data = {
    'images': dev_videos,
    'annotations': dev_annotations
}

with open('/content/drive/MyDrive/thesis-data/dev_annotations_50.json', 'w') as f:
    json.dump(dev_data, f)

print("Dev annotation file saved to Drive!")

Dev videos: 50
Dev captions: 1000
Dev annotation file saved to Drive!


In [30]:
import os

videos_on_drive = os.listdir('/content/drive/MyDrive/thesis-data/videos')
print(f"Videos already on Drive: {len(videos_on_drive)}")
print("Sample:", videos_on_drive[:5])

Videos already on Drive: 0
Sample: []


In [31]:
import os

all_videos = sorted(os.listdir('/content/msrvtt_extracted/TrainValVideo/'))
print(f"Total videos available: {len(all_videos)}")
print("First 5:", all_videos[:5])

Total videos available: 7010
First 5: ['video0.mp4', 'video1.mp4', 'video10.mp4', 'video100.mp4', 'video1000.mp4']


In [32]:
import shutil

video_source = '/content/msrvtt_extracted/TrainValVideo/'
drive_videos = '/content/drive/MyDrive/thesis-data/videos/'

# Take first 50 videos
videos_to_copy = sorted(all_videos)[:50]

print(f"Copying {len(videos_to_copy)} videos...")

for i, vid in enumerate(videos_to_copy):
    src = os.path.join(video_source, vid)
    dst = os.path.join(drive_videos, vid)
    shutil.copy(src, dst)
    if (i+1) % 10 == 0:
        print(f"  {i+1}/50 done...")

print("All 50 videos copied to Drive!")

Copying 50 videos...
  10/50 done...
  20/50 done...
  30/50 done...
  40/50 done...
  50/50 done...
All 50 videos copied to Drive!


In [33]:
videos_on_drive = os.listdir('/content/drive/MyDrive/thesis-data/videos/')
print(f"Videos on Drive: {len(videos_on_drive)}")
print("Sample:", sorted(videos_on_drive)[:5])

Videos on Drive: 50
Sample: ['video0.mp4', 'video1.mp4', 'video10.mp4', 'video100.mp4', 'video1000.mp4']


In [34]:
import shutil, os

video_source = '/content/msrvtt_extracted/TrainValVideo/'
drive_videos = '/content/drive/MyDrive/thesis-data/videos/'

# Clear what's already there
for f in os.listdir(drive_videos):
    os.remove(os.path.join(drive_videos, f))

# Copy video0.mp4 to video49.mp4 specifically
videos_to_copy = [f'video{i}.mp4' for i in range(50)]

missing = []
for vid in videos_to_copy:
    src = os.path.join(video_source, vid)
    dst = os.path.join(drive_videos, vid)
    if os.path.exists(src):
        shutil.copy(src, dst)
    else:
        missing.append(vid)

print(f"Copied: {50 - len(missing)} videos")
print(f"Missing: {missing}")

Copied: 50 videos
Missing: []


In [35]:
videos_on_drive = sorted(os.listdir('/content/drive/MyDrive/thesis-data/videos/'),
                         key=lambda x: int(x.replace('video','').replace('.mp4','')))

print(f"Total videos: {len(videos_on_drive)}")
print("First 5:", videos_on_drive[:5])
print("Last 5:", videos_on_drive[-5:])

Total videos: 50
First 5: ['video0.mp4', 'video1.mp4', 'video2.mp4', 'video3.mp4', 'video4.mp4']
Last 5: ['video45.mp4', 'video46.mp4', 'video47.mp4', 'video48.mp4', 'video49.mp4']


In [37]:
import shutil

shutil.copy(
    '/content/msrvtt_annotation/MSR_VTT.json',
    '/content/drive/MyDrive/thesis-data/MSR_VTT.json'
)
print("Done!")

Done!


In [38]:
import os
print(os.path.exists('/content/drive/MyDrive/thesis-data/MSR_VTT.json'))  # Should print True

True
